## 05 Tool Calling with MCP
MCP (or Model Context Protocol) creates a open standard protocol between tools and other applications. The basic operation is similar to the standard tool flow in LangChain. The model is provided with the tool description as before, but now it is done via signaling with the MCP server to get the descriptions. And, rather than executing the tool in the tool node, execution takes place on the MCP server when requested by the agent.

Here is a diagram that shows you how:

<div align="center">
<img src="images/05_tools_with_mcp.png" width="450" heigh="500" alt="Tools with MCP"/>
</div>

MCP provides a standardized way to connect AI Agents with external tools and data sources. We can use LangChain MCP Adapters to help us connect to MCP servers for LangChain Agents.

| <div style="text-align: center">**⚠️ WINDOWS ISSUE**</div> |
| :--- |
| **This notebook will not run on Windows 11!!!** <br/> The issue stems from a three-way architectural conflict between Windows 11, the MCP library, and Jupyter’s virtualization. Windows requires the Proactor event loop to handle subprocess communication, but this loop is often incompatible with Jupyter’s existing event loop. Jupyter replaces standard system streams (stderr/stdout) with "virtual" objects to display output in your browser. MCP attempts to access the low-level system handle (fileno) of these streams to monitor the server. Because Jupyter’s virtual streams lack these handles, the code triggers a terminal UnsupportedOperation error that cannot be easily bypassed within the Notebook environment. |

In [1]:
import sys, os
import asyncio
from dotenv import load_dotenv
from rich.console import Console
from rich.markdown import Markdown
from typing import Literal, Union

from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langchain.tools import tool

load_dotenv(override=True)
console = Console()

> **NOTE**:
>
> There is a **known Windows + Jupyter notebook incompatibility issue** when MCP tries to launch subprocess-based servers, like we will do in this notebook.<br/><br/>
>
> Run the following cell **BEFORE** making any MCP calls on Windows and when running inside a Notebook.<br/><br/>

In [2]:
# Fix for Windows issues in Jupyter notebooks
if sys.platform == "win32":
    # 1. Use ProactorEventLoop for subprocess support
    if not isinstance(
        asyncio.get_event_loop_policy(), asyncio.WindowsProactorEventLoopPolicy
    ):
        asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())

    # 2. Redirect stderr to avoid fileno() error when launching MCP servers
    if "ipykernel" in sys.modules:
        sys.stderr = sys.__stderr__

In [ ]:
from langchain_mcp_adapters.client import MultiServerMCPClient

client2 = MultiServerMCPClient(
    {
        "time": {
            "transport": "stdio",
            "command": "npx.cmd",  # Always use .cmd on Windows
            "args": ["-y", "@theo.foobar/mcp-time"],
        }
    },
)

client = MultiServerMCPClient(
    {
        "time": {
            "transport": "stdio",
            "command": "uvx",
            "args": ["mcp-server-time", "--local-timezone=America/New_York"],
        }
    }
)

In [11]:
tools = await client.get_tools()
print("-------- list of tools available -------- ")
console.print(tools)

-------- list of tools available -------- 


[
    StructuredTool(
        name='get_current_time',
        description='Get current time in a specific timezones',
        args_schema={
            'type': 'object',
            'properties': {
                'timezone': {
                    'type': 'string',
                    'description': "IANA timezone name (e.g., 'America/New_York', 'Europe/London'). Use 
'America/New_York' as local timezone if no timezone provided by the user."
                }
            },
            'required': ['timezone']
        },
        response_format='content_and_artifact',
        coroutine=<function convert_mcp_tool_to_langchain_tool.<locals>.call_tool at 0x000001CB7103DA80>
    ),
    StructuredTool(
        name='convert_time',
        description='Convert time between timezones',
        args_schema={
            'type': 'object',
            'properties': {
                'source_timezone': {
                    'type': 'string',
                    'description': "Source IANA timezone name (e.g., 'America/New_York', 'Europe/London'). Use 
'America/New_York' as local timezone if no source timezone provided by the user."
                },
                'time': {'type': 'string', 'description': 'Time to convert in 24-hour format (HH:MM)'},
                'target_timezone': {
                    'type': 'string',
                    'description': "Target IANA timezone name (e.g., 'Asia/Tokyo', 'America/San_Francisco'). Use 
'America/New_York' as local timezone if no target timezone provided by the user."
                }
            },
            'required': ['source_timezone', 'time', 'target_timezone']
        },
        response_format='content_and_artifact',
        coroutine=<function convert_mcp_tool_to_langchain_tool.<locals>.call_tool at 0x000001CB7103DDA0>
    )
]

In [5]:
import asyncio
import sys
import os
import nest_asyncio
from langchain_mcp_adapters.client import MultiServerMCPClient

# 1. Essential for Windows Subprocesses
if sys.platform == "win32":
    asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())

nest_asyncio.apply()


async def run_mcp():
    # 2. Redirect stderr to the 'null' device
    # This prevents the 'fileno' error in Jupyter environments
    f = open(os.devnull, "w")
    original_stderr = sys.stderr
    sys.stderr = f

    try:
        mcp_client = MultiServerMCPClient(
            {
                "time": {
                    "transport": "stdio",
                    "command": "npx.cmd",  # Always use .cmd on Windows
                    "args": ["-y", "@theo.foobar/mcp-time"],
                }
            },
        )

        # 3. New API (v0.1.0+) - No 'async with' on the client itself
        mcp_tools = await mcp_client.get_tools()

        # Switch back to original stderr to see our print results
        sys.stderr = original_stderr
        print(f"✅ Success! Loaded tools: {[tool.name for tool in mcp_tools]}")

    except Exception as e:
        sys.stderr = original_stderr
        print(f"❌ Failed: {e}")
    finally:
        f.close()


# Run the execution
await run_mcp()

❌ Failed: fileno


In [4]:
# lets define our MCP tool connector
# Please pip install langchain-mcp-adapters to use this tool
from langchain_mcp_adapters.client import MultiServerMCPClient
import nest_asyncio

# Fix for Windows
import asyncio
import sys

# 1. Force the ProactorEventLoop for Windows subprocess support
if sys.platform == "win32":
    asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())

nest_asyncio.apply()


async def run_mcp():
    # 2. Configure the client
    # We use npx.cmd for Windows compatibility
    mcp_client = MultiServerMCPClient(
        {
            "time": {
                "transport": "stdio",
                "command": "npx.cmd",
                "args": ["-y", "@theo.foobar/mcp-time"],
            }
        },
    )

    try:
        # 3. Use the direct method as suggested by the error message
        # Note: If this still throws the 'fileno' error in Jupyter,
        # it is because the internal mcp library is trying to log stderr.
        mcp_tools = await mcp_client.get_tools()
        print(f"Successfully loaded: {[tool.name for tool in mcp_tools]}")

    except Exception as e:
        print(f"Error: {e}")


# Run the async function in your cell
await run_mcp()

# Workaround for Windows + Jupyter:
#
# mcp's stdio_client captures sys.stderr as the default `errlog` at *import time*,
# before Jupyter replaces it with its own OutStream. When MCP spawns a subprocess it
# passes that OutStream as `stderr=errlog` to subprocess.Popen, which calls
# errlog.fileno() — but Jupyter's OutStream has no real file descriptor and raises
# UnsupportedOperation: fileno.
#
# Fix: monkey-patch the internal Windows fallback process creator so that any
# errlog that lacks a working fileno() is silently replaced with subprocess.PIPE.


# if sys.platform == "win32":
#     import mcp.os.win32.utilities as _mcp_win32

#     _orig_fallback = _mcp_win32._create_windows_fallback_process

#     async def _patched_fallback(command, args, env, errlog, cwd):
#         try:
#             if errlog is not None:
#                 errlog.fileno()
#         except Exception:
#             errlog = subprocess.PIPE
#         return await _orig_fallback(command, args, env, errlog, cwd)

#     _mcp_win32._create_windows_fallback_process = _patched_fallback

# mcp_client = MultiServerMCPClient(
#     {
#         "time": {
#             "transport": "stdio",
#             "command": "npx",
#             "args": ["-y", "@theo.foobar/mcp-time"],
#         }
#     },
# )
# mcp_tools = await mcp_client.get_tools()
# print(f"Loaded {len(mcp_tools)} -> {[tool.name for tool in mcp_tools]}")

Error: fileno
